

# Delta Live Tables


<div  style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://raw.githubusercontent.com/derar-alhussein/Databricks-Certified-Data-Engineer-Associate/main/Includes/images/bookstore_schema.png" alt="Databricks Learning" style="width: 600">
</div>


>> Documentaion

In [0]:
%sql
/*
CREATE OR REFRESH [TEMPORARY] { STREAMING TABLE | MATERIALIZED VIEW } table_name [CLUSTER BY (col_name1, col_name2, ... )]
  [(
    [
    col_name1 col_type1 [ GENERATED ALWAYS AS generation_expression1 ] [ COMMENT col_comment1 ] [ column_constraint ] [ MASK func_name [ USING COLUMNS ( other_column_name | constant_literal [, ...] ) ] ],
    col_name2 col_type2 [ GENERATED ALWAYS AS generation_expression2 ] [ COMMENT col_comment2 ] [ column_constraint ] [ MASK func_name [ USING COLUMNS ( other_column_name | constant_literal [, ...] ) ] ],
    ...
    ]
    [
    CONSTRAINT expectation_name_1 EXPECT (expectation_expr1) [ON VIOLATION { FAIL UPDATE | DROP ROW }],
    CONSTRAINT expectation_name_2 EXPECT (expectation_expr2) [ON VIOLATION { FAIL UPDATE | DROP ROW }],
    ...
    ]
    [ table_constraint ] [, ...]
  )]
  [USING DELTA]
  [PARTITIONED BY (col_name1, col_name2, ... )]
  [LOCATION path]
  [COMMENT table_comment]
  [TBLPROPERTIES (key1 [ = ] val1, key2 [ = ] val2, ... )]
  [ WITH { ROW FILTER func_name ON ( [ column_name | constant_literal [, ...] ] ) [...] } ]
  AS select_statement

  */


In [0]:
%run ../Includes/Copy-Datasets

In [0]:
%sql
-- drop schema if exists demo_books_store_dlt_db CASCADE;

In [0]:
%python
# remove folder path destination pipeline bookstore 
path = 'dbfs:/mnt/demo'

dbutils.fs.rm(path, True)

In [0]:
SET datasets.path=dbfs:/mnt/demo-datasets/bookstore;

In [0]:
-- SELECT * FROM json.`${datasets.path}/orders-json-raw`;

Databricks data profile. Run in Databricks to view.

In [0]:
-- SELECT * FROM json.`${datasets.path}/customers-json`;

Databricks data profile. Run in Databricks to view.

## Bronze Layer Tables

#### orders_raw

In [0]:
-- TRAITEMENT INCREMENTIAL (AUTO LOADER) NECESSITE L'AJOUT DU MOT CLE STREAMING 
CREATE OR REFRESH STREAMING LIVE TABLE orders_raw
COMMENT "The raw books orders, ingested from orders-raw"
AS SELECT * FROM cloud_files("${datasets.path}/orders-json-raw", "json", 
                             map("cloudFiles.inferColumnTypes", "true"))

#### customers

In [0]:
CREATE OR REFRESH LIVE TABLE customers
COMMENT "The customers lookup table, ingested from customers-json"
AS SELECT * FROM json.`${datasets.path}/customers-json`




## Silver Layer Tables

#### orders_cleaned

In [0]:
CREATE OR REFRESH STREAMING LIVE TABLE orders_cleaned (
  CONSTRAINT valid_order_number EXPECT (order_id IS NOT NULL) ON VIOLATION DROP ROW
)
COMMENT "The cleaned books orders with valid order_id"
AS
  SELECT order_id, quantity, o.customer_id, c.profile:first_name as f_name, c.profile:last_name as l_name,
         cast(from_unixtime(order_timestamp, 'yyyy-MM-dd HH:mm:ss') AS timestamp) order_timestamp, o.books,
         c.profile:address:country as country
  FROM STREAM(LIVE.orders_raw) o
  LEFT JOIN LIVE.customers c
    ON o.customer_id = c.customer_id

>> Constraint violation

| **`ON VIOLATION`** | Behavior |
| --- | --- |
| **`DROP ROW`** | Discard records that violate constraints |
| **`FAIL UPDATE`** | Violated constraint causes the pipeline to fail  |
| Omitted | Records violating constraints will be kept, and reported in metrics |



## Gold Tables

In [0]:
CREATE OR REFRESH LIVE TABLE cn_daily_customer_books
COMMENT "Daily number of books per customer in China"
AS
  SELECT customer_id, f_name, l_name, date_trunc("DD", order_timestamp) order_date, sum(quantity) books_counts
  FROM LIVE.orders_cleaned
  WHERE country = "China"
  GROUP BY customer_id, f_name, l_name, date_trunc("DD", order_timestamp)

>> Add files in order_raw streming table and seeing in pipeline 

In [0]:
%python
# # Define the datasets.path variable
# datasets_path = "dbfs:/mnt/demo-datasets/bookstore"

# # Use the resolved path
# files = dbutils.fs.ls(f"{datasets_path}/orders-json-raw")
# display(files)

In [0]:
CREATE OR REFRESH LIVE TABLE fr_daily_customer_books
COMMENT "Daily number of books per customer in France"
AS
  SELECT customer_id, f_name, l_name, date_trunc("DD", order_timestamp) order_date, sum(quantity) books_counts
  FROM LIVE.orders_cleaned
  WHERE country = "France"
  GROUP BY customer_id, f_name, l_name, date_trunc("DD", order_timestamp)

In [0]:
%python
files = dbutils.fs.ls('dbfs:/mnt/demo/dlt_booksstore')
display(files)

In [0]:
%python
files = dbutils.fs.ls('dbfs:/mnt/demo/dlt_booksstore/system/events/')
display(files)